# Fase 5: Simulasi Komputasi (Monte Carlo, Bloom Filter, MCMC)
**Member:** Fernando — Computation Analyst

**Pertanyaan Riset (P3):** Berapa probabilitas terjadinya keterlambatan penyelesaian *issue* (*bottleneck* > 30 hari) diestimasi tanpa rumus analitik, dan bagaimana cara memfilter duplikasi pelaporan serta mengoptimasi beban *Sprint* para *maintainer*?

**Tujuan:** Menggunakan Monte Carlo untuk estimasi probabilitas empiris, merancang struktur *Bloom Filter* untuk mengecek duplikasi *issue*, serta mengaplikasikan simulasi stokastik *Markov Chain Monte Carlo* (MCMC) pada *Knapsack Problem* untuk *Sprint Planning*.

## AI Usage Disclosure
**Member:** Fernando — Computation Analyst | **Tools used:** ChatGPT

| Task | Tool | Prompt summary | Output modified? |
| --- | --- | --- | --- |
| Referensi hash unik | ChatGPT | "Bagaimana cara generate multiple hash index yang unik di Python untuk Bloom Filter menggunakan satu library bawaan?" | Ya — disesuaikan untuk perulangan `hashlib.md5`. |

**Written entirely without AI:** Seluruh logika penerapan iterasi Monte Carlo, perumusan FPR *Bloom Filter* secara manual, modifikasi algoritma *Metropolis-Hastings* untuk batasan kapasitas *Knapsack*, serta seluruh penulisan interpretasi naratif di dokumen ini dikerjakan 100% mandiri tanpa bantuan AI.

In [11]:
import sys
import numpy as np
import pandas as pd

sys.path.append("..")
# Import fungsi dari layer komputasi milik Member E
from src.simulation import estimate_probability, BloomFilter, mcmc_knapsack

# Load data bersih hasil ekstraksi Member A
df = pd.read_csv("../data/clean/dataset.csv")
df['created_at'] = pd.to_datetime(df['created_at'])
df['closed_at'] = pd.to_datetime(df['closed_at'])

# Hitung durasi penyelesaian dalam jam
df['time_to_close_hours'] = (df['closed_at'] - df['created_at']).dt.total_seconds() / 3600
durations = df.dropna(subset=['time_to_close_hours'])['time_to_close_hours'].values

### 1. Estimasi Probabilitas Bottleneck (Monte Carlo Simulation)
Kita ingin mengetahui seberapa besar ancaman penumpukan *issue* yang memakan waktu penyelesaian lebih dari 30 hari (720 jam). Karena distribusi waktu dunia nyata jarang mengikuti kurva normal sempurna, kita akan menyimulasikan 50.000 iterasi stokastik berdasarkan data empiris.

In [18]:
def is_bottleneck():
    """Fungsi event: mengembalikan True jika issue memakan waktu > 720 jam."""
    sample = np.random.choice(durations)
    return sample > 720.0

# Eksekusi Monte Carlo
prob_bottleneck = estimate_probability(is_bottleneck, n_trials=50000)
print(f"Probabilitas terjadinya bottleneck (>30 hari) berdasarkan 50,000 iterasi: {prob_bottleneck:.4f} ({prob_bottleneck*100:.2f}%)")

Probabilitas terjadinya bottleneck (>30 hari) berdasarkan 50,000 iterasi: 0.0905 (9.05%)


### 2. Efisiensi Caching Duplikasi dengan Bloom Filter
Dengan ratusan laporan masuk setiap minggunya, *maintainer* butuh filter untuk menekan *issue* duplikat. Kami menguji struktur probabilistik *Bloom Filter* untuk mendeteksi apakah sebuah ID *issue* sudah pernah diproses.


In [9]:
# Inisiasi array memori 10000 bits dengan 5 fungsi hash
bf = BloomFilter(k=5, m=10000)
n_issues_simulated = 2000

# Masukkan 2000 issue unik ke dalam filter
for i in range(n_issues_simulated):
    bf.add(f"issue_pandas_{i}")

# Kalkulasi False Positive Rate (FPR) teoretis
fpr = bf.theoretical_fpr(n=n_issues_simulated)
print(f"Theoretical False Positive Rate (n={n_issues_simulated}, m=10000, k=5) : {fpr:.5f} ({fpr*100:.2f}%)")

# Uji Coba Pengecekan
print(f"Cek 'issue_pandas_1500' (seharusnya True)  : {bf.contains('issue_pandas_1500')}")
print(f"Cek 'issue_pandas_9999' (seharusnya False) : {bf.contains('issue_pandas_9999')}")

Theoretical False Positive Rate (n=2000, m=10000, k=5) : 0.00020 (0.02%)
Cek 'issue_pandas_1500' (seharusnya True)  : True
Cek 'issue_pandas_9999' (seharusnya False) : False


### 3. Simulasi MCMC Knapsack untuk Sprint Planning
Sebagai bentuk optimasi manajerial, kita menyimulasikan alokasi pekerjaan untuk rilis *Sprint* ke depan. Asumsikan *maintainer* hanya memiliki kapasitas kerja **1000 jam**. Terdapat 30 *issue* mendesak dengan beban jam dan "Nilai Dampak/Value" yang berbeda. Kita mencari kombinasi beban terbaik menggunakan MCMC.

In [10]:
np.random.seed(42)
# Generate 30 issue dummy: (Beban Waktu dalam jam, Nilai Dampak)
sprint_issues = [(np.random.randint(10, 100), np.random.randint(1, 10)) for _ in range(30)]
max_capacity = 1000.0

# Jalankan 100,000 iterasi MCMC
best_combination, total_value = mcmc_knapsack(sprint_issues, max_capacity, n_iter=100000)

total_weight = sum(sprint_issues[i][0] for i in range(len(sprint_issues)) if best_combination[i])

print(f"Kapasitas Maksimal Sprint     : {max_capacity} jam")
print(f"Total Beban Waktu Terpilih    : {total_weight} jam")
print(f"Total Nilai Dampak (Optimasi) : {total_value}")

Kapasitas Maksimal Sprint     : 1000.0 jam
Total Beban Waktu Terpilih    : 893 jam
Total Nilai Dampak (Optimasi) : 132


### Kesimpulan Akhir Analisis (Penutup)
Modul Komputasi ini resmi menutup seluruh rangkaian Audit Statistik. Pertanyaan Riset 3 (P3) telah terjawab; simulasi Monte Carlo berhasil mengestimasi probabilitas empiris *bottleneck*, sementara Bloom Filter dan MCMC terbukti mampu memberikan solusi arsitektural dan manajerial stokastik untuk menekan inefisiensi beban kerja *maintainer*. Seluruh temuan dari Modul 01 hingga 05 siap untuk ditranslasikan ke dalam bentuk *Statistical Health Report* final.
